# Exercise 17: Error Handling

Explore **common failure modes** (invalid model, bad schema, context overflow)
and learn the **resilience patterns** (retry with exponential backoff) needed
for production OpenAI integrations.

In [ ]:
import json

from dotenv import load_dotenv
from openai import OpenAI, APIError, RateLimitError, BadRequestError, AuthenticationError

load_dotenv()
client = OpenAI()

## Error test helper

A utility that runs a callable and pretty-prints the exception details.

In [ ]:
def test_error(name, func):
    """Run a function and display the error details."""
    print(f"\n--- {name} ---")
    try:
        func()
        print("  (No error -- unexpected!)")
    except RateLimitError as e:
        print(f"  Error type: RateLimitError (HTTP {e.status_code})")
        print(f"  Message: {e.message[:120]}")
        print(f"  Retry strategy: Exponential backoff, check Retry-After header")
    except BadRequestError as e:
        print(f"  Error type: BadRequestError (HTTP {e.status_code})")
        print(f"  Message: {e.message[:200]}")
        print(f"  Fix: Check your request parameters")
    except AuthenticationError as e:
        print(f"  Error type: AuthenticationError (HTTP {e.status_code})")
        print(f"  Message: {e.message[:120]}")
        print(f"  Fix: Check your API key")
    except APIError as e:
        print(f"  Error type: {type(e).__name__} (HTTP {e.status_code})")
        print(f"  Message: {e.message[:200]}")
    except Exception as e:
        print(f"  Error type: {type(e).__name__}")
        print(f"  Message: {str(e)[:200]}")

## Common error modes

Each cell deliberately triggers a different class of API error.

In [ ]:
# 1. Invalid model name
test_error("Invalid model name", lambda: client.responses.create(
    model="gpt-5-turbo-nonexistent",
    input="Hello",
))

In [ ]:
# 2. Malformed function schema (missing required field)
test_error("Malformed function schema", lambda: client.responses.create(
    model="gpt-4.1-mini",
    input="Hello",
    tools=[{
        "type": "function",
        "name": "bad_func",
        # Missing "parameters" and "strict"
    }],
))

In [ ]:
# 3. Invalid structured output schema
test_error("Invalid structured output schema", lambda: client.responses.create(
    model="gpt-4.1-mini",
    input="Hello",
    text={"format": {
        "type": "json_schema",
        "name": "test",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                # With strict, additionalProperties: False is required
            },
        },
    }},
))

In [ ]:
# 4. Context window overflow (simulate with huge input)
test_error("Very long input (context test)", lambda: client.responses.create(
    model="gpt-4.1-mini",
    input="x " * 500_000,  # Way over context limit
    max_output_tokens=10,
))

In [ ]:
# 5. Empty input
test_error("Empty input", lambda: client.responses.create(
    model="gpt-4.1-mini",
    input="",
))

## Resilience pattern: Retry with exponential backoff

In [ ]:
print("""
from openai import OpenAI, RateLimitError
import time

client = OpenAI()

def call_with_retry(func, max_retries=3, base_delay=1):
    for attempt in range(max_retries):
        try:
            return func()
        except RateLimitError as e:
            if attempt == max_retries - 1:
                raise
            delay = base_delay * (2 ** attempt)  # 1s, 2s, 4s
            print(f"Rate limited. Retrying in {delay}s...")
            time.sleep(delay)

# Usage:
# result = call_with_retry(
#     lambda: client.responses.create(model="gpt-4.1-mini", input="Hello")
# )
""")

## Error handling cheat sheet

In [ ]:
print("Error                  | HTTP | Cause                    | Action")
print("-----------------------|------|--------------------------|---------------------------")
print("AuthenticationError    | 401  | Bad API key              | Check OPENAI_API_KEY")
print("RateLimitError         | 429  | Too many requests        | Exponential backoff")
print("BadRequestError        | 400  | Invalid params/schema    | Fix the request")
print("NotFoundError          | 404  | Invalid model/resource   | Check model name")
print("APIError               | 500  | Server-side issue        | Retry after delay")
print("APIConnectionError     | -    | Network issue            | Check connectivity, retry")
print()
print("Key tips for enterprise:")
print("- Always use try/except around API calls")
print("- Implement exponential backoff for 429s")
print("- Log error types and messages for debugging")
print("- Set max_output_tokens to prevent runaway costs")
print("- Use strict: true schemas to catch errors at compile time, not runtime")

## Interview one-liner

> Wrap every OpenAI call in `try/except`, catch `RateLimitError` for exponential backoff and `BadRequestError` for schema bugs -- and always set `max_output_tokens` so a runaway generation cannot blow your budget.